# Phase 3 - NEURON to MuJoCo

Profile-driven workflow (recommended):
1. Load Phase 2 spikes + mapping
2. Build actuator controls
3. Apply a named signal profile
4. Auto-map controls to MuJoCo ctrlrange
5. Render video

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import yaml

PHASE3_ROOT = Path('/path/to/local/digifly-master/Phase 3').expanduser().resolve()
SRC_DIR = PHASE3_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from phase3_bridge import (
    load_phase2_spike_times,
    load_phase2_timebase_ms,
    load_mapping_csv,
    summarize_mapping_coverage,
    build_actuator_controls_from_spikes,
    save_controls_csv,
    plot_actuator_controls,
    apply_profile_transforms,
    render_controls_mujoco,
)

print('PHASE3_ROOT =', PHASE3_ROOT)

In [ ]:
# ---------- Required inputs ----------
PHASE2_RUN_DIR = Path('/path/to/local/digifly-master/Phase 2/data/export_swc/hemi_runs/run_debug_full').expanduser().resolve()
MAPPING_CANDIDATES = [
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_with_groups.csv',
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_updated.csv',
    PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping.csv',
]
MAPPING_CSV = next((p.resolve() for p in MAPPING_CANDIDATES if p.exists()), None)
if MAPPING_CSV is None:
    raise FileNotFoundError(f'No mapping CSV found. Checked: {[str(p) for p in MAPPING_CANDIDATES]}')
PROFILE_YAML = (PHASE3_ROOT / 'configs' / 'phase3_video_profiles.yaml').resolve()

# Activation kernel (spike -> control precursor)
TAU_RISE_MS = 1.0
TAU_DECAY_MS = 6.0
GLOBAL_SCALE = 1.0

# Choose profile name; None uses default_profile from YAML
PROFILE_NAME = None

# Run tag controls output folder names
RUN_TAG = 'run_debug_full_bridge'

# Render toggle
RENDER_ENABLED = True

# Candidate MuJoCo world XMLs (first existing path is used)
WORLD_XML_CANDIDATES = [
    PHASE3_ROOT / 'data' / 'inputs' / 'flybody' / 'floor.xml',
    Path('/path/to/local/flybody-main/flybody/fruitfly/assets/floor.xml'),
]

WORLD_XML = next((p.expanduser().resolve() for p in WORLD_XML_CANDIDATES if p.expanduser().resolve().exists()), None)

DERIVED_DIR = (PHASE3_ROOT / 'data' / 'derived' / RUN_TAG).resolve()
FIG_DIR = (PHASE3_ROOT / 'data' / 'outputs' / 'figures').resolve()
VID_DIR = (PHASE3_ROOT / 'data' / 'outputs' / 'videos').resolve()

print('PHASE2_RUN_DIR =', PHASE2_RUN_DIR)
print('MAPPING_CSV    =', MAPPING_CSV)
print('PROFILE_YAML   =', PROFILE_YAML)
print('WORLD_XML      =', WORLD_XML)


In [ ]:
with open(PROFILE_YAML, 'r') as f:
    profile_doc = yaml.safe_load(f)

if PROFILE_NAME is None:
    PROFILE_NAME = str(profile_doc.get('default_profile', 'balanced_showcase'))

profiles = profile_doc.get('profiles', {})
if PROFILE_NAME not in profiles:
    raise KeyError(f'Profile not found: {PROFILE_NAME}. Available: {sorted(profiles)}')

PROFILE = profiles[PROFILE_NAME]
print('PROFILE_NAME =', PROFILE_NAME)
print(json.dumps(PROFILE, indent=2))

In [ ]:
spikes = load_phase2_spike_times(PHASE2_RUN_DIR)
t_ms = load_phase2_timebase_ms(PHASE2_RUN_DIR)
mapping = load_mapping_csv(MAPPING_CSV)

coverage = summarize_mapping_coverage(spikes, mapping)
print('[coverage]')
for k, v in coverage.items():
    print(f'  {k}: {v}')

controls_base_df, build_stats = build_actuator_controls_from_spikes(
    spikes=spikes,
    mapping=mapping,
    t_ms=t_ms,
    tau_rise_ms=TAU_RISE_MS,
    tau_decay_ms=TAU_DECAY_MS,
    scale=GLOBAL_SCALE,
)

base_act = {c: controls_base_df[c].to_numpy(dtype=float) for c in controls_base_df.columns if c != 't_ms'}
t_mod, mod_act = apply_profile_transforms(
    controls_base_df['t_ms'].to_numpy(dtype=float),
    base_act,
    PROFILE.get('signal', {}),
)

controls_mod_df = pd.DataFrame({'t_ms': t_mod})
for k in sorted(mod_act):
    controls_mod_df[k] = mod_act[k]

DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
VID_DIR.mkdir(parents=True, exist_ok=True)

base_csv = save_controls_csv(controls_base_df, DERIVED_DIR / f'controls_base_{PROFILE_NAME}.csv')
mod_csv = save_controls_csv(controls_mod_df, DERIVED_DIR / f'controls_profiled_{PROFILE_NAME}.csv')

base_fig = plot_actuator_controls(controls_base_df, FIG_DIR / f'{RUN_TAG}_{PROFILE_NAME}_base_top12.pdf', top_n=12)
mod_fig = plot_actuator_controls(controls_mod_df, FIG_DIR / f'{RUN_TAG}_{PROFILE_NAME}_profiled_top12.pdf', top_n=12)

summary = {
    'profile_name': PROFILE_NAME,
    **coverage,
    **build_stats,
    'base_controls_csv': str(base_csv),
    'profiled_controls_csv': str(mod_csv),
    'base_plot': str(base_fig),
    'profiled_plot': str(mod_fig),
}

summary_json = DERIVED_DIR / f'bridge_summary_{PROFILE_NAME}.json'
with open(summary_json, 'w') as f:
    json.dump(summary, f, indent=2)

print('[save] base controls ->', base_csv)
print('[save] profiled controls ->', mod_csv)
print('[save] base figure ->', base_fig)
print('[save] profiled figure ->', mod_fig)
print('[save] summary ->', summary_json)

In [ ]:
render_report_path = DERIVED_DIR / f'render_report_{PROFILE_NAME}.json'

if not RENDER_ENABLED:
    print('[render] skipped (RENDER_ENABLED=False)')
elif WORLD_XML is None:
    print('[render] skipped (no WORLD_XML found in WORLD_XML_CANDIDATES)')
else:
    render_cfg = PROFILE.get('render', {})
    video_out = VID_DIR / f'{RUN_TAG}_{PROFILE_NAME}.mp4'
    try:
        render_report = render_controls_mujoco(
            mjcf_xml=WORLD_XML,
            t_ms=controls_mod_df['t_ms'].to_numpy(dtype=float),
            act_controls={c: controls_mod_df[c].to_numpy(dtype=float) for c in controls_mod_df.columns if c != 't_ms'},
            out_video=video_out,
            camera_name=str(render_cfg.get('camera_name', 'track2')),
            camera_distance_factor=float(render_cfg.get('camera_distance_factor', 8.0)),
            camera_fovy_deg=float(render_cfg.get('camera_fovy_deg', 75.0)),
            render_hz=int(render_cfg.get('render_hz', 120)),
            slowmo=float(render_cfg.get('slowmo', 20.0)),
            width=int(render_cfg.get('width', 1280)),
            height=int(render_cfg.get('height', 720)),
        )
        with open(render_report_path, 'w') as f:
            json.dump(render_report, f, indent=2)
        print('[render] video ->', video_out)
        print('[render] report ->', render_report_path)
        print('[render] stats:', render_report)
    except Exception as e:
        err = {'error': str(e)}
        with open(render_report_path, 'w') as f:
            json.dump(err, f, indent=2)
        print('[render] failed:', e)
        print('[render] wrote error report ->', render_report_path)


## Why this is better than legacy manual tuning
- Signal transforms are explicit and saved by profile name.
- Control scaling is auto-calibrated per actuator against model `ctrlrange`.
- Camera/render choices are isolated in the profile (no hidden state).
- Outputs are versioned by `RUN_TAG` + `PROFILE_NAME` in `data/derived` and `data/outputs`.